In [15]:
import jax
import jax.numpy as jnp
import optax
import time
import qutip as qt
import numpy as np

from qst_tec.basicfunc import custom_nuclear_norm, build_separable_state, distance_sq, partial_transpose, ppt_penalty, ppt_value
# 配置使用双精度
jax.config.update("jax_enable_x64", True)

# ==========================================
# 第1部分：verifier section

In [16]:
# 1. 验证器 Loss：只求最小化几何距离
@jax.jit
def verifier_loss(params_V, rho_target):
    rho_sep = build_separable_state(params_V)
    return distance_sq(rho_target, rho_sep)


# ==========================================
# 第二部分：构造待验证的量子态

In [ ]:
d = 4

# 1. 定义单体 4 维空间的计算基矢
k0 = qt.basis(d, 0)
k1 = qt.basis(d, 1)
k2 = qt.basis(d, 2)

# 2. 优雅地构造 5 个 UPB 直积态 (.unit() 会自动处理分母的 sqrt(2) 或 sqrt(3))
v0 = qt.tensor(k0, (k0 - k1).unit())
v1 = qt.tensor((k0 - k1).unit(), k2)
v2 = qt.tensor(k2, (k1 - k2).unit())
v3 = qt.tensor((k1 - k2).unit(), k0)
v4 = qt.tensor((k0 + k1 + k2).unit(), (k0 + k1 + k2).unit())

# 3. 构造 UPB 投影算符之和 (使用 .proj() 直接获得态的密度矩阵投影)
P_UPB = v0.proj() + v1.proj() + v2.proj() + v3.proj() + v4.proj()

# 4. 构造前 3x3 子空间内的单位阵 (利用列表推导式求和)
I_9 = sum([qt.tensor(qt.basis(d, i), qt.basis(d, j)).proj() 
           for i in range(3) for j in range(3)])

# 5. 在正交补空间中生成束缚纠缠态
rho_bes_qutip = 0.98* (I_9 - P_UPB) / 4.0 +  0.02*qt.tensor(qt.qeye(4), qt.qeye(4)) / 16.0

# 6. 一键转回 JAX 数组，供您的梯度下降或层析算法使用
rho_target = jnp.array(rho_bes_qutip.full(), dtype=jnp.complex64)
rho_target = rho_target = jnp.array([
    [0.25, 0.0, 0.0, 0.0,   0.0, 0.25, 0.0, 0.0,   0.0, 0.0, 0.25, 0.0,   0.0, 0.0, 0.0, 0.25],
    [0.0,  0.0, 0.0, 0.0,   0.0, 0.0,  0.0, 0.0,   0.0, 0.0, 0.0,  0.0,   0.0, 0.0, 0.0, 0.0 ],
    [0.0,  0.0, 0.0, 0.0,   0.0, 0.0,  0.0, 0.0,   0.0, 0.0, 0.0,  0.0,   0.0, 0.0, 0.0, 0.0 ],
    [0.0,  0.0, 0.0, 0.0,   0.0, 0.0,  0.0, 0.0,   0.0, 0.0, 0.0,  0.0,   0.0, 0.0, 0.0, 0.0 ],

    [0.0,  0.0, 0.0, 0.0,   0.0, 0.0,  0.0, 0.0,   0.0, 0.0, 0.0,  0.0,   0.0, 0.0, 0.0, 0.0 ],
    [0.25, 0.0, 0.0, 0.0,   0.0, 0.25, 0.0, 0.0,   0.0, 0.0, 0.25, 0.0,   0.0, 0.0, 0.0, 0.25],
    [0.0,  0.0, 0.0, 0.0,   0.0, 0.0,  0.0, 0.0,   0.0, 0.0, 0.0,  0.0,   0.0, 0.0, 0.0, 0.0 ],
    [0.0,  0.0, 0.0, 0.0,   0.0, 0.0,  0.0, 0.0,   0.0, 0.0, 0.0,  0.0,   0.0, 0.0, 0.0, 0.0 ],

    [0.0,  0.0, 0.0, 0.0,   0.0, 0.0,  0.0, 0.0,   0.0, 0.0, 0.0,  0.0,   0.0, 0.0, 0.0, 0.0 ],
    [0.0,  0.0, 0.0, 0.0,   0.0, 0.0,  0.0, 0.0,   0.0, 0.0, 0.0,  0.0,   0.0, 0.0, 0.0, 0.0 ],
    [0.25, 0.0, 0.0, 0.0,   0.0, 0.25, 0.0, 0.0,   0.0, 0.0, 0.25, 0.0,   0.0, 0.0, 0.0, 0.25],
    [0.0,  0.0, 0.0, 0.0,   0.0, 0.0,  0.0, 0.0,   0.0, 0.0, 0.0,  0.0,   0.0, 0.0, 0.0, 0.0 ],

    [0.0,  0.0, 0.0, 0.0,   0.0, 0.0,  0.0, 0.0,   0.0, 0.0, 0.0,  0.0,   0.0, 0.0, 0.0, 0.0 ],
    [0.0,  0.0, 0.0, 0.0,   0.0, 0.0,  0.0, 0.0,   0.0, 0.0, 0.0,  0.0,   0.0, 0.0, 0.0, 0.0 ],
    [0.0,  0.0, 0.0, 0.0,   0.0, 0.0,  0.0, 0.0,   0.0, 0.0, 0.0,  0.0,   0.0, 0.0, 0.0, 0.0 ],
    [0.25, 0.0, 0.0, 0.0,   0.0, 0.25, 0.0, 0.0,   0.0, 0.0, 0.25, 0.0,   0.0, 0.0, 0.0, 0.25]
], dtype=jnp.complex64)
print(rho_target)

[[ 0.09652778+0.j  0.09527778+0.j -0.02722222+0.j  0.        +0.j
  -0.02722222+0.j -0.02722222+0.j -0.02722222+0.j  0.        +0.j
  -0.02722222+0.j -0.02722222+0.j -0.02722222+0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j]
 [ 0.09527778+0.j  0.09652778+0.j -0.02722222+0.j  0.        +0.j
  -0.02722222+0.j -0.02722222+0.j -0.02722222+0.j  0.        +0.j
  -0.02722222+0.j -0.02722222+0.j -0.02722222+0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j]
 [-0.02722222+0.j -0.02722222+0.j  0.09652778+0.j  0.        +0.j
  -0.02722222+0.j -0.02722222+0.j  0.09527778+0.j  0.        +0.j
  -0.02722222+0.j -0.02722222+0.j -0.02722222+0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j]
 [ 0.        +0.j  0.        +0.j  0.        +0.j  0.00125   +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
   0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j
   0.  

In [18]:
# ==========================================
# 第三部分：JIT 编译双方的更新步 (交替更新)

In [19]:
# ==========================================

lr_V = 0.05
opt_V = optax.adam(lr_V)


@jax.jit
def update_verifier(params_V, opt_state_V, rho_target):
    loss_val, grads = jax.value_and_grad(verifier_loss)(params_V, rho_target)
    # JAX 复数梯度坑：对 Tuple 里的每一个矩阵取共轭
    grads = jax.tree.map(lambda g: jnp.conj(g), grads)
    updates, new_opt_state = opt_V.update(grads, opt_state_V, params_V)
    new_params = optax.apply_updates(params_V, updates)
    return new_params, new_opt_state, loss_val

In [20]:
# ==========================================
# 第四部分：开始对抗训练 (The Game)

In [21]:
# ==========================================

def verifier_separable_state(d=4, K=None, steps=1000, rho_verified=rho_target):
    # Carathéodory bound: K_max = rank_T² is the theoretical maximum
    # for pure product states in an R-dimensional subspace.
    if d is None:
        d = int(rho_verified.shape[0] ** 0.5)

    if K is None:
        K = int(rho_verified.shape[0] **2)

    key = jax.random.PRNGKey(42)
    k1, k2, k3, k4 = jax.random.split(key, 4)

    # 随机初始化双方参数
    A = jax.random.normal(k1, (K, d)) + 1j * jax.random.normal(k2, (K, d))
    B = jax.random.normal(k3, (K, d)) + 1j * jax.random.normal(k4, (K, d))
    params_V = (A, B)
    
    
    opt_state_V = opt_V.init(params_V)
    
    print(f"开始搜索可分态分解: d={d}, K={K}")
    start_time = time.time()
    
    for i in range(steps):
            params_V, opt_state_V, v_loss = update_verifier(params_V, opt_state_V, rho_verified)
        
            if i % 100 == 0:
              print(f"Step {i:4d} | 几何距离: {v_loss:.5f} ")
            
    print(f"耗时: {time.time() - start_time:.2f}s")
    return params_V



# K is auto-computed from rank_T via the Carathéodory bound (K = rank_T²).
final_V = verifier_separable_state(d=4, K=16**2, steps=2000, rho_verified=rho_target)


# 打印它的 CCNR 看看！
print("最终态的 与可分态的Frobenius 范数平方距离:", verifier_loss(final_V, rho_target))
print("最终态的 density matrix:", build_separable_state(final_V))
print("最终态的 ppt value:", ppt_penalty(build_separable_state(final_V)))
print("目标态的 ppt value:", ppt_value(rho_target))

开始搜索可分态分解: d=4, K=256
Step    0 | 几何距离: 0.18588 
Step  100 | 几何距离: 0.00173 
Step  200 | 几何距离: 0.00148 
Step  300 | 几何距离: 0.00142 
Step  400 | 几何距离: 0.00140 
Step  500 | 几何距离: 0.00139 
Step  600 | 几何距离: 0.00139 
Step  700 | 几何距离: 0.00138 
Step  800 | 几何距离: 0.00138 
Step  900 | 几何距离: 0.00138 
Step 1000 | 几何距离: 0.00138 
Step 1100 | 几何距离: 0.00138 
Step 1200 | 几何距离: 0.00138 
Step 1300 | 几何距离: 0.00138 
Step 1400 | 几何距离: 0.00138 
Step 1500 | 几何距离: 0.00138 
Step 1600 | 几何距离: 0.00138 
Step 1700 | 几何距离: 0.00138 
Step 1800 | 几何距离: 0.00138 
Step 1900 | 几何距离: 0.00138 
耗时: 0.66s
最终态的 与可分态的Frobenius 范数平方距离: 0.0013779675152332505
最终态的 density matrix: [[ 9.91795160e-02-4.33719847e-19j  8.65041824e-02-3.49216744e-09j
  -2.22589337e-02-3.49021303e-08j -8.34192972e-08-5.46451398e-08j
  -2.58747281e-02+7.85821290e-09j -2.48741514e-02+2.87981927e-09j
  -2.67174423e-02+2.86635876e-08j  1.95759984e-08+1.43610420e-08j
  -2.22583637e-02+7.16207346e-09j -2.67208854e-02+3.98298672e-09j
  -1.88516865e-02+6.2124838